# Mobius transforms and invariance

In [72]:
from sympy import *

# -- Symbols --

(x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, n2, T) = symbols(
    '''x, y, z, t, x0, y0, z0, z1, t0, Z, g2, g3, m, n, l, k, i, j, M, N, n2, T'''
)
(delta, nu, Aeff) = symbols(
    '''delta, nu, Aeff'''
)

gtilde2 = Symbol('gtilde2', latex_name=r'\tilde{g_2}')
gtilde3 = Symbol('gtilde3', latex_name=r'\tilde{g_3}')

# -- Functions --

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
zeta = Function('zeta') # Weierstrass Zeta function

rhop = Function('\\rho\'')
Delta = Function('Delta')
rho = Function('rho')
kappa = Function('kappa')
phi = Function('phi')
h = Function('h')
q = Function('q')
s = Function('s')
u = Function('u')
v = Function('v')
w = Function('w')
ws = Function('ws')
chi = Function('chi')
xi = Function('xi')
P = Function('P') # Polynomial
Q = Function('Q') # Polynomial
R = Function('R') # Polynomial

U = Function('U')
V = Function('V')
W = Function('W')
H = Function('H')


uhat = Function('uhat', latex_name=r'\hat{u}')
vhat = Function('vhat', latex_name=r'\hat{v}')
Hhat = Symbol('Hhat', latex_name=r'\hat{H}')

ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')
Hbar = Symbol('Hbar', latex_name=r'\bar{H}')

utilde = Function('utilde', latex_name=r'\tilde{u}')
vtilde = Function('vtilde', latex_name=r'\tilde{v}')
Htilde = Symbol('Htilde', latex_name=r'\tilde{H}')
htilde = Function('htilde', latex_name=r'\tilde{h}')
wtilde = Function('wtilde', latex_name=r'\tilde{w}')


wp = Function('wp')


Summ = Function('Summ')

# -- Indexed Symbols --

Omega = IndexedBase('Omega')
A = IndexedBase('A')
C = IndexedBase('C')
F = IndexedBase('F')
r = IndexedBase('r')
gamma = IndexedBase('gamma')
mu = IndexedBase('mu')
nu = IndexedBase('nu')
theta = IndexedBase('theta')
X = IndexedBase('X')
Y = IndexedBase('Y')
a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
G = IndexedBase('G')
psi = IndexedBase('psi')
Psi = IndexedBase('Psi')
upsilon = IndexedBase('upsilon')
epsilon = IndexedBase('epsilon')
omega = IndexedBase('omega')
alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
K = IndexedBase('K')
lam = IndexedBase('lambda')
Lam = IndexedBase('Lambda')

wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# kth order derivatives of Weierstrass P
from wpk import wpk, wzk, wsk, run_tests

# The package containing mpmath expressions for Weierstrass elliptic functions
from numerical_evaluation.weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp

# Numeric solutions to diff eqs
from numpy import linspace, absolute, angle, square, real, imag, conj, array as arraynp, concatenate
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
import scipy.integrate
import matplotlib.pyplot as plt
from random import random

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Modal power transforms

We want to explore modal power transforms and answer the following:
- If the corss-ratio is preserved does it imply there is a Mobius transform between the modes?
- Does the answer to the above chnage if the modal powers are themselves connected by conservation laws such as intermodal power conservation?
- When is the derivative of the modal power invariant under the transfroms being considered?

### Transform definition

In [87]:
u_uhat = Eq(u(z, mu[j]), sqrt(a[j])*uhat(z, mu[j])/sqrt(h(z)))
v_vhat = Eq(v(z, mu[j]), sqrt(a[j])*vhat(z, mu[j])/sqrt(h(z)))
uv_uvhat = Eq(u_uhat.lhs*v_vhat.lhs, u_uhat.rhs*v_vhat.rhs)
uv_gam_rho = Eq(u(z, mu[j])*v(z, mu[j]), gamma[j] - rho(z))
uv_gam_cons = Eq(
    uv_gam_rho.lhs - uv_gam_rho.lhs.subs(j,k),
    uv_gam_rho.rhs - uv_gam_rho.rhs.subs(j,k)
)
sum_gam_0 = Eq(Sum(gamma[j],(j,1,N)), 0)

u_uhat
v_vhat
uv_uvhat
uv_gam_rho
uv_gam_cons
sum_gam_0

Eq(u(z, mu[j]), uhat(z, mu[j])*sqrt(a[j])/sqrt(h(z)))

Eq(v(z, mu[j]), vhat(z, mu[j])*sqrt(a[j])/sqrt(h(z)))

Eq(u(z, mu[j])*v(z, mu[j]), uhat(z, mu[j])*vhat(z, mu[j])*a[j]/h(z))

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]), gamma[j] - gamma[k])

Eq(Sum(gamma[j], (j, 1, N)), 0)

## Theorem 1: Assuming intermodal power is conserved

### Lemma 1: If the new modes preserve intermodal power conservation, $h(z)$ must be the reciprocal of a linear function of $\rho(z)$, and thus the new mode powers must be a Mobius transform of the old mode powers

Suppose that for some constants $b_j, b_k, j\ne k, b_j \ne 0 $, we have intermodal power conservation in the new modes such that we can write for some constants $\psi_j, \psi_k$:

In [32]:
uvhat_assume_pow_cons = Eq(
    uv_uvhat.rhs*h(z)*b[j] - (uv_uvhat.rhs*h(z)*b[j]).subs(j,k),
    psi[j] - psi[k]
)

uvhat_assume_pow_cons

Eq(uhat(z, mu[j])*vhat(z, mu[j])*a[j]*b[j] - uhat(z, mu[k])*vhat(z, mu[k])*a[k]*b[k], psi[j] - psi[k])

We can always write:

In [37]:
# Always true
uvhat_pow_diff_1 = Eq(
    uv_uvhat.lhs*h(z)*b[j] - (uv_uvhat.lhs*h(z)*b[j]).subs(j,k),
    uv_uvhat.rhs*h(z)*b[j] - (uv_uvhat.rhs*h(z)*b[j]).subs(j,k)
)
uvhat_pow_diff_1

Eq(h(z)*u(z, mu[j])*v(z, mu[j])*b[j] - h(z)*u(z, mu[k])*v(z, mu[k])*b[k], uhat(z, mu[j])*vhat(z, mu[j])*a[j]*b[j] - uhat(z, mu[k])*vhat(z, mu[k])*a[k]*b[k])

and thus, if the intermodal power conservation holds for $\hat{u}, \hat{v}$ we must have:

In [62]:
# Consequence of assumption
uvhat_pow_diff_assume = uvhat_pow_diff_1.subs([uvhat_assume_pow_cons.args])
uvhat_pow_diff_assume = Eq(uvhat_pow_diff_assume.lhs.factor(), uvhat_pow_diff_assume.rhs)
uvhat_pow_diff_assume_b = uvhat_pow_diff_assume.subs([uv_gam_rho.args, uv_gam_rho.subs(j,k).args])
uvhat_pow_diff_assume_h_rho = Eq(h(z), solve(uvhat_pow_diff_assume_b, h(z))[0])
cjk = Eq(c[j,k], (b[j]*gamma[j] - b[k]*gamma[k])/(psi[j] - psi[k]))
djk = Eq(d[j,k], (b[j] - b[k])/(psi[j] - psi[k]))
uvhat_pow_diff_assume_h_rho_cd = uvhat_pow_diff_assume_h_rho.subs([
    (gamma[j], solve(cjk, gamma[j])[0])
]).simplify().subs([
    (psi[j], solve(djk, psi[j])[0])
]).simplify()


uvhat_pow_diff_assume
uvhat_pow_diff_assume_h_rho
cjk
djk
uvhat_pow_diff_assume_h_rho_cd

Eq((u(z, mu[j])*v(z, mu[j])*b[j] - u(z, mu[k])*v(z, mu[k])*b[k])*h(z), psi[j] - psi[k])

Eq(h(z), (-psi[j] + psi[k])/(rho(z)*b[j] - rho(z)*b[k] - b[j]*gamma[j] + b[k]*gamma[k]))

Eq(c[j, k], (b[j]*gamma[j] - b[k]*gamma[k])/(psi[j] - psi[k]))

Eq(d[j, k], (b[j] - b[k])/(psi[j] - psi[k]))

Eq(h(z), 1/(-rho(z)*d[j, k] + c[j, k]))

where we note that $c_{j,k}\ne 0$ and $d_{j,k}\ne 0$ else that would imply that $b_j=0$ contrary to our assumptions. It follows that we must have a Mobius transform between the new and old modal powers of the form:

In [44]:
uvhat_mobius_uv = Eq(
    h(z)*uv_uvhat.rhs/a[j], 
    (
        (h(z)*uv_uvhat.lhs/a[j]).subs([uvhat_pow_diff_assume_h_rho.args])
        .subs(rho(z), solve(uv_gam_rho, rho(z))[0])
    ).simplify()
)
uvhat_mobius_uv = Eq(
    uvhat_mobius_uv.lhs,
    fraction(uvhat_mobius_uv.rhs)[0]/
    (fraction(uvhat_mobius_uv.rhs)[1]).collect(u(z, mu[j])*v(z, mu[j]), factor)
)
uvhat_mobius_uv

Eq(uhat(z, mu[j])*vhat(z, mu[j]), (psi[j] - psi[k])*u(z, mu[j])*v(z, mu[j])/(((b[j] - b[k])*u(z, mu[j])*v(z, mu[j]) + (gamma[j] - gamma[k])*b[k])*a[j]))

### Lemma 2: The relationships between intermodal power constants in different coordinates

If $h(z)$ is related to $\rho(z)$ vai Mobius transform that transfrom must be unique for non constant $\rho(z)$. It follows that there exists constants $A_0, A_1$ such that $c_{j,k}=A_0\ne 0$ and $d_{j,k}=A_1\ne 0$.

In [98]:
rho_dc = Eq(
    uvhat_pow_diff_assume_h_rho_cd.lhs -
    uvhat_pow_diff_assume_h_rho_cd.lhs.subs([(j,l),(k,m)]),
    uvhat_pow_diff_assume_h_rho_cd.rhs -
    uvhat_pow_diff_assume_h_rho_cd.rhs.subs([(j,l),(k,m)])
)
rho_dc = Eq(
    rho_dc.lhs*fraction(rho_dc.rhs.simplify())[1],
    (rho_dc.rhs*fraction(rho_dc.rhs.simplify())[1]).simplify().expand().collect(rho(z), factor)
)
cjk = Eq(c[j,k], solve(rho_dc.rhs.coeff(rho(z),0), c[j,k])[0])
djk = Eq(d[j,k], solve(rho_dc.rhs.coeff(rho(z),1), d[j,k])[0])
cjk_A = Eq(cjk.lhs, A[0])
djk_A = Eq(djk.lhs, A[1])
h_rho_A = uvhat_pow_diff_assume_h_rho_cd.subs([cjk_A.args, djk_A.args])

uvhat_pow_diff_assume_h_rho_cd
rho_dc
cjk
djk
cjk_A
djk_A
h_rho_A

Eq(h(z), 1/(-rho(z)*d[j, k] + c[j, k]))

Eq(0, (d[j, k] - d[l, m])*rho(z) - c[j, k] + c[l, m])

Eq(c[j, k], c[l, m])

Eq(d[j, k], d[l, m])

Eq(c[j, k], A[0])

Eq(d[j, k], A[1])

Eq(h(z), 1/(-rho(z)*A[1] + A[0]))

In [85]:
uv_j_uvhat = Eq(
    u(z,mu[j])*v(z, mu[j]), 
    solve(
        uv_uvhat.subs([h_rho_A.args, (rho(z), solve(uv_gam_rho, rho(z))[0])]),
        u(z,mu[j])*v(z, mu[j])
    )[0]
)

gamma_jk_uvhat = Eq(
    (uv_j_uvhat.lhs - uv_j_uvhat.lhs.subs(j,k)).subs([uv_gam_cons.args]),
    uv_j_uvhat.rhs - uv_j_uvhat.rhs.subs(j,k)
)

uvhat_jk = Eq(
    uhat(z, mu[k])*vhat(z, mu[k]),
    solve(gamma_jk_uvhat, uhat(z, mu[k])*vhat(z, mu[k]))[0]
    .expand().collect(uhat(z, mu[j])*vhat(z, mu[j]), factor)

)
   
uv_j_uvhat
gamma_jk_uvhat
uvhat_jk

Eq(u(z, mu[j])*v(z, mu[j]), (-A[0] + A[1]*gamma[j])*uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(uhat(z, mu[j])*vhat(z, mu[j])*A[1]*a[j] - 1))

Eq(gamma[j] - gamma[k], -(-A[0] + A[1]*gamma[k])*uhat(z, mu[k])*vhat(z, mu[k])*a[k]/(uhat(z, mu[k])*vhat(z, mu[k])*A[1]*a[k] - 1) + (-A[0] + A[1]*gamma[j])*uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(uhat(z, mu[j])*vhat(z, mu[j])*A[1]*a[j] - 1))

Eq(uhat(z, mu[k])*vhat(z, mu[k]), (A[0] - A[1]*gamma[k])*uhat(z, mu[j])*vhat(z, mu[j])*a[j]/((A[0] - A[1]*gamma[j])*a[k]) - (gamma[j] - gamma[k])/((A[0] - A[1]*gamma[j])*a[k]))

### Lemma 3: The definition of $h(z)$ in $\hat{u}\hat{v}$

It follows that, if intermodal power conservation is preserved under the transformation then $h(z)$ must be a weighted sum of modal powers plus a constant.

In [97]:
mean_uv_uvhat = Eq(
    Sum(uv_uvhat.lhs,(j,1,N))/N, 
    Sum(fraction(uv_uvhat.rhs)[0],(j,1,N))/fraction(uv_uvhat.rhs)[1]/N
)
mean_uv_rho = Eq(
    Sum(uv_uvhat.lhs,(j,1,N))/N,
    -rho(z)
)

h_sum_uvhat = Eq(
    h(z), 
    solve(
        mean_uv_uvhat.subs([mean_uv_rho.args, (rho(z), solve(h_rho_A, rho(z))[0])]),
        h(z)
    )[0].expand()
)

mean_uv_uvhat
mean_uv_rho
h_sum_uvhat

Eq(Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, N))/N, Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/(N*h(z)))

Eq(Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, N))/N, -rho(z))

Eq(h(z), 1/A[0] - A[1]*Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/(N*A[0]))

Thus we have shown that this sum form for $h(z)$ is neccessary for intermodal power conservation to be preserved under the transformation.

### Lemma 4: Sufficient form of $h(z)$ to preserve intermodal power conservation

Suppose that $h(z)$ is of the following form with $a_0 \ne 0, a_j \ne 0, \Omega_0 \ne 0$:

In [281]:
h_aj = Eq(h(z), a[0] + Omega[0]/N*Sum(a[j]*uhat(z, mu[j])*vhat(z, mu[j]), (j,1,N)))

h_aj

Eq(h(z), a[0] + Omega[0]*Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/N)

It follows that $\rho(z)$ is related to $h(z)$ via a Mobius transform:

In [282]:
sum_uv_uvhat = Eq(
    Sum(uv_uvhat.lhs, (j,1,N)), 
    Sum(fraction(uv_uvhat.rhs)[0],(j,1,N))/fraction(uv_uvhat.rhs)[1]
)
sum_uv_uvhat_rho_h = Eq(
    -N*rho(z), 
    sum_uv_uvhat.rhs.subs([
        (
            Sum(fraction(uv_uvhat.rhs)[0],(j,1,N)),
            solve(h_aj, Sum(fraction(uv_uvhat.rhs)[0],(j,1,N)))[0]
        )
    ])
)
rho_h_suff = Eq(
    rho(z), solve(sum_uv_uvhat_rho_h, rho(z))[0]
)
h_rho_suff = Eq(
    h(z), solve(sum_uv_uvhat_rho_h, h(z))[0]
)

sum_uv_uvhat
rho_h_suff
h_rho_suff

Eq(Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, N)), Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/h(z))

Eq(rho(z), (-h(z) + a[0])/(h(z)*Omega[0]))

Eq(h(z), a[0]/(rho(z)*Omega[0] + 1))

It follows that the modal powers in the new coordinates are related to the old coordinates via Mobius transform:

In [283]:
uvhat_uv_suff_mobius = Eq(
    uhat(z, mu[j])*vhat(z, mu[j]),
    solve(
        uv_uvhat.subs([h_rho_suff.args, (rho(z), solve(uv_gam_rho, rho(z))[0])]),
        uhat(z, mu[j])*vhat(z, mu[j])
    )[0]
)
uv_uvhat_suff_mobius = Eq(
    u(z, mu[j])*v(z, mu[j]),
    solve(
        uvhat_uv_suff_mobius,
        u(z, mu[j])*v(z, mu[j])
    )[0]
)


uvhat_uv_suff_mobius
uv_uvhat_suff_mobius

Eq(uhat(z, mu[j])*vhat(z, mu[j]), u(z, mu[j])*v(z, mu[j])*a[0]/((-u(z, mu[j])*v(z, mu[j])*Omega[0] + Omega[0]*gamma[j] + 1)*a[j]))

Eq(u(z, mu[j])*v(z, mu[j]), (Omega[0]*gamma[j] + 1)*uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(uhat(z, mu[j])*vhat(z, mu[j])*Omega[0]*a[j] + a[0]))

and that intermodal power conservation laws exist in the new coordinates in the form:

In [285]:
gam_jk_suff_uvhat = Eq(
    (uv_uvhat_suff_mobius.lhs - uv_uvhat_suff_mobius.subs(j,k).lhs).subs([
        uv_gam_rho.args, uv_gam_rho.subs(j,k).args
    ]),
    uv_uvhat_suff_mobius.rhs - uv_uvhat_suff_mobius.subs(j,k).rhs
)

_cc = Omega[0]*gamma[k]+1
uvhatk_uvhatj_suff = Eq(
    uhat(z, mu[k])*vhat(z, mu[k])*a[k]/_cc,
    (
        solve(
            gam_jk_suff_uvhat,
            uhat(z, mu[k])*vhat(z, mu[k])*a[k]
        )[0]/_cc
    ).expand().collect(uhat(z, mu[j])*vhat(z, mu[j]), factor)
)

uvhatk_uvhatj_suff = Eq(
    uvhatk_uvhatj_suff.lhs - uvhatk_uvhatj_suff.rhs.subs(a[0],0),
    sum(_.factor() for _ in apart(uvhatk_uvhatj_suff.rhs - uvhatk_uvhatj_suff.rhs.subs(a[0],0), gamma[k]).args)
)


gam_jk_suff_uvhat
uvhatk_uvhatj_suff

Eq(gamma[j] - gamma[k], (Omega[0]*gamma[j] + 1)*uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(uhat(z, mu[j])*vhat(z, mu[j])*Omega[0]*a[j] + a[0]) - (Omega[0]*gamma[k] + 1)*uhat(z, mu[k])*vhat(z, mu[k])*a[k]/(uhat(z, mu[k])*vhat(z, mu[k])*Omega[0]*a[k] + a[0]))

Eq(uhat(z, mu[k])*vhat(z, mu[k])*a[k]/(Omega[0]*gamma[k] + 1) - uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(Omega[0]*gamma[j] + 1), -a[0]/((Omega[0]*gamma[k] + 1)*Omega[0]) + a[0]/((Omega[0]*gamma[j] + 1)*Omega[0]))

### Lemma 5: $h(z)$ is linear in any single mode in the new coordinates

In [286]:
h_uvhat_j = Eq(
    (uv_uvhat.lhs - uv_uvhat.lhs.subs(j,k)).subs([uv_gam_cons.args]),
    (uv_uvhat.rhs - uv_uvhat.rhs.subs(j,k)).subs([
        (uhat(z, mu[k])*vhat(z, mu[k]), solve(uvhatk_uvhatj_suff, uhat(z, mu[k])*vhat(z, mu[k]))[0])
    ])
)
h_uvhat_j = Eq(h(z), solve(h_uvhat_j, h(z))[0].expand().collect(uhat(z, mu[j])*vhat(z, mu[j]), factor))

h_uvhat_j

Eq(h(z), uhat(z, mu[j])*vhat(z, mu[j])*Omega[0]*a[j]/(Omega[0]*gamma[j] + 1) + a[0]/(Omega[0]*gamma[j] + 1))

## Theorem 2: Without assuming intermodal power is conserved

### Lemma 1: Only two modal powers at most are linearly independent

In [287]:
uvhat_gam_jkl = Eq(
    (uv_uvhat.lhs - uv_uvhat.lhs.subs(j,k)).subs([uv_gam_cons.args])/
    (uv_uvhat.lhs - uv_uvhat.lhs.subs(j,l)).subs([uv_gam_cons.subs(k,l).args]),
    (
        (uv_uvhat.rhs - uv_uvhat.rhs.subs(j,k))/
        (uv_uvhat.rhs - uv_uvhat.rhs.subs(j,l))
    ).simplify()
)
# k != j, l != j
uvhat_j_gam_kl_ind = Eq(
    uhat(z, mu[j])*vhat(z, mu[j]),
    solve(uvhat_gam_jkl, uhat(z, mu[j])*vhat(z, mu[j]))[0]
    .expand().collect([uhat(z, mu[k])*vhat(z, mu[k]), uhat(z, mu[l])*vhat(z, mu[l])], factor)
)

uvhat_gam_jkl
uvhat_j_gam_kl_ind

Eq((gamma[j] - gamma[k])/(gamma[j] - gamma[l]), (uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[k])*vhat(z, mu[k])*a[k])/(uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[l])*vhat(z, mu[l])*a[l]))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), -(gamma[j] - gamma[k])*uhat(z, mu[l])*vhat(z, mu[l])*a[l]/((gamma[k] - gamma[l])*a[j]) + (gamma[j] - gamma[l])*uhat(z, mu[k])*vhat(z, mu[k])*a[k]/((gamma[k] - gamma[l])*a[j]))

### Lemma 2: $h(z)$ can always be expressed as a weighted sum of modal powers in the new coordinates

In [288]:
rho_sum_uvhat_a = Eq(
    Sum(gamma[j],(j,1,N)) - rho(z)*N,
    Sum(uv_uvhat.rhs,(j,1,N))
)
rho_sum_uvhat_b = Eq(
    rho(z),
    -Sum(fraction(uv_uvhat.rhs)[0],(j,1,N))/N/fraction(uv_uvhat.rhs)[1]
)
uv_j_sum_uvhat = Eq(
    uv_uvhat.lhs - gamma[j],
    Sum(fraction(uv_uvhat.rhs)[0],(j,1,N))/N/fraction(uv_uvhat.rhs)[1]
)
uvhat_j_sum_uvhat = Eq(
    uv_uvhat.rhs*fraction(uv_uvhat.rhs)[1],
    gamma[j]*fraction(uv_uvhat.rhs)[1] + Sum(fraction(uv_uvhat.rhs)[0],(j,1,N))/N
)
h_sum_uvhat = Eq(
    h(z),
    Sum(
        uhat(z, mu[j])*vhat(z, mu[j])*a[j]*( 1/gamma[j] - Sum(1/gamma[k], (k,1,N))/N ), 
        (j, 1, N)
    )/N,
)

sum_uv_uvhat
rho_sum_uvhat_a
rho_sum_uvhat_b
uv_j_sum_uvhat
uvhat_j_sum_uvhat
h_sum_uvhat

Eq(Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, N)), Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/h(z))

Eq(-N*rho(z) + Sum(gamma[j], (j, 1, N)), Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j]/h(z), (j, 1, N)))

Eq(rho(z), -Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/(N*h(z)))

Eq(u(z, mu[j])*v(z, mu[j]) - gamma[j], Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/(N*h(z)))

Eq(uhat(z, mu[j])*vhat(z, mu[j])*a[j], h(z)*gamma[j] + Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/N)

Eq(h(z), Sum((1/gamma[j] - Sum(1/gamma[k], (k, 1, N))/N)*uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/N)

### Lemma 3: The weighted cross-ratio is preserved and constant

In [289]:
cr_uv_uvhat = Eq(
    (uv_uvhat.lhs - uv_uvhat.lhs.subs(j,k))*(uv_uvhat.lhs.subs(j,l) - uv_uvhat.lhs.subs(j,m))/
    ((uv_uvhat.lhs.subs(j,l) - uv_uvhat.lhs.subs(j,k))*(uv_uvhat.lhs - uv_uvhat.lhs.subs(j,m))),
    (
        (uv_uvhat.rhs - uv_uvhat.rhs.subs(j,k))*(uv_uvhat.rhs.subs(j,l) - uv_uvhat.rhs.subs(j,m))/
        ((uv_uvhat.rhs.subs(j,l) - uv_uvhat.rhs.subs(j,k))*(uv_uvhat.rhs - uv_uvhat.rhs.subs(j,m)))
    ).factor()
)
cr_uv_uvhat_const = cr_uv_uvhat.subs([
    uv_gam_cons.args,
    uv_gam_cons.subs([(k,m)]).args,
    uv_gam_cons.subs([(j,l),(k,m)]).args,
    uv_gam_cons.subs([(j,l)]).args
])

cr_uv_uvhat
cr_uv_uvhat_const

Eq((u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]))*(u(z, mu[l])*v(z, mu[l]) - u(z, mu[m])*v(z, mu[m]))/((u(z, mu[j])*v(z, mu[j]) - u(z, mu[m])*v(z, mu[m]))*(-u(z, mu[k])*v(z, mu[k]) + u(z, mu[l])*v(z, mu[l]))), -(uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[k])*vhat(z, mu[k])*a[k])*(uhat(z, mu[l])*vhat(z, mu[l])*a[l] - uhat(z, mu[m])*vhat(z, mu[m])*a[m])/((uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[m])*vhat(z, mu[m])*a[m])*(uhat(z, mu[k])*vhat(z, mu[k])*a[k] - uhat(z, mu[l])*vhat(z, mu[l])*a[l])))

Eq((gamma[j] - gamma[k])*(gamma[l] - gamma[m])/((gamma[j] - gamma[m])*(-gamma[k] + gamma[l])), -(uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[k])*vhat(z, mu[k])*a[k])*(uhat(z, mu[l])*vhat(z, mu[l])*a[l] - uhat(z, mu[m])*vhat(z, mu[m])*a[m])/((uhat(z, mu[j])*vhat(z, mu[j])*a[j] - uhat(z, mu[m])*vhat(z, mu[m])*a[m])*(uhat(z, mu[k])*vhat(z, mu[k])*a[k] - uhat(z, mu[l])*vhat(z, mu[l])*a[l])))

In [290]:
cr_uv_uvhat_const.subs([
    (k,1),
    (l,2),
    uvhat_j_gam_kl_ind.subs([(k,1), (l,2)]).args,
    uvhat_j_gam_kl_ind.subs([(j,m), (k,1), (l,2)]).args
]).simplify()

True

### Lemma 4: Intermodal power ratios are preserved

In [291]:
uv_uvhat_ratio = Eq(
    uv_uvhat.lhs/uv_uvhat.lhs.subs(j,k),
    uv_uvhat.rhs/uv_uvhat.rhs.subs(j,k)
)

uv_uvhat_ratio

Eq(u(z, mu[j])*v(z, mu[j])/(u(z, mu[k])*v(z, mu[k])), uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(uhat(z, mu[k])*vhat(z, mu[k])*a[k]))

## Derivative of the transform assuming intermodal power conservation

As before, we consider the following starting properties in $u,v$, and the following transform to $\hat{u},\hat{v}$:

In [292]:
u_uhat
v_vhat
uv_uvhat
uv_gam_rho
uv_gam_cons
sum_gam_0


Eq(u(z, mu[j]), uhat(z, mu[j])*sqrt(a[j])/sqrt(h(z)))

Eq(v(z, mu[j]), vhat(z, mu[j])*sqrt(a[j])/sqrt(h(z)))

Eq(u(z, mu[j])*v(z, mu[j]), uhat(z, mu[j])*vhat(z, mu[j])*a[j]/h(z))

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(u(z, mu[j])*v(z, mu[j]) - u(z, mu[k])*v(z, mu[k]), gamma[j] - gamma[k])

Eq(Sum(gamma[j], (j, 1, N)), 0)

We assume that intermodal power conservation remains in the new coordinates and thus without loss of generality we assume the following form and properties for $h(z)$:

In [293]:
uvhat_h_j = Eq(uhat(z, mu[j])*vhat(z, mu[j]), solve(h_uvhat_j, uhat(z, mu[j])*vhat(z, mu[j]))[0])

h_aj
uvhatk_uvhatj_suff
rho_h_suff
h_rho_suff
h_uvhat_j
uvhat_h_j

Eq(h(z), a[0] + Omega[0]*Sum(uhat(z, mu[j])*vhat(z, mu[j])*a[j], (j, 1, N))/N)

Eq(uhat(z, mu[k])*vhat(z, mu[k])*a[k]/(Omega[0]*gamma[k] + 1) - uhat(z, mu[j])*vhat(z, mu[j])*a[j]/(Omega[0]*gamma[j] + 1), -a[0]/((Omega[0]*gamma[k] + 1)*Omega[0]) + a[0]/((Omega[0]*gamma[j] + 1)*Omega[0]))

Eq(rho(z), (-h(z) + a[0])/(h(z)*Omega[0]))

Eq(h(z), a[0]/(rho(z)*Omega[0] + 1))

Eq(h(z), uhat(z, mu[j])*vhat(z, mu[j])*Omega[0]*a[j]/(Omega[0]*gamma[j] + 1) + a[0]/(Omega[0]*gamma[j] + 1))

Eq(uhat(z, mu[j])*vhat(z, mu[j]), (h(z)*Omega[0]*gamma[j] + h(z) - a[0])/(Omega[0]*a[j]))

We will explore how the derivative behaves under transformation, assuming that it solves the following differential equation in the orginal coordinates:

In [294]:
duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]),z),
    Product(u(z, mu[j]),(j,1,N)) - Product(v(z, mu[j]),(j,1,N))
)
d_rho_uvj = Eq(
    Derivative(rho(z),z),
    -Product(u(z, mu[j]),(j,1,N)) + Product(v(z, mu[j]),(j,1,N))
)
duvj
d_rho_uvj

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), Product(u(z, mu[j]), (j, 1, N)) - Product(v(z, mu[j]), (j, 1, N)))

Eq(Derivative(rho(z), z), -Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))

In [295]:
drho_dh = Eq(diff(rho_h_suff.lhs,z), diff(rho_h_suff.rhs,z).simplify())
dh_drho = Eq(diff(h_rho_suff.lhs,z), diff(h_rho_suff.rhs,z).simplify())
d_rho_uvhatj = Eq(
    Derivative(rho(z),z),
    sqrt(Product(a[j],(j,1,N)))*
    (-Product(uhat(z, mu[j]),(j,1,N)) + Product(vhat(z, mu[j]),(j,1,N)))/h(z)**(N/2)
)

drho_dh
dh_drho
d_rho_uvhatj

Eq(Derivative(rho(z), z), -Derivative(h(z), z)*a[0]/(h(z)**2*Omega[0]))

Eq(Derivative(h(z), z), -Derivative(rho(z), z)*Omega[0]*a[0]/(rho(z)*Omega[0] + 1)**2)

Eq(Derivative(rho(z), z), (-Product(uhat(z, mu[j]), (j, 1, N)) + Product(vhat(z, mu[j]), (j, 1, N)))*sqrt(Product(a[j], (j, 1, N)))/h(z)**(N/2))

In [296]:
Eq(Derivative(uv_uvhat.lhs,z), Derivative(uv_uvhat.rhs,z))
Eq(
    Derivative(a[j]*uhat(z, mu[j])*vhat(z, mu[j])/h(z), z),
    a[j]*Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z)/h(z) +
    a[j]*uhat(z, mu[j])*vhat(z, mu[j])*diff(1/h(z), z)
)

duvhat_rho = Eq(
    Derivative(gamma[j]-rho(z), z).doit(),
    (
        a[j]*Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z)/h(z) +
        a[j]*uhat(z, mu[j])*vhat(z, mu[j])*diff(1/h(z), z)
    ).subs(diff(h(z),z), solve(drho_dh, diff(h(z),z))[0])
)

duvhat_rho = Eq(
    Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z),
    solve(duvhat_rho, Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z))[0]
)

duvhat_rho_b = Eq(
    duvhat_rho.lhs,
    duvhat_rho.rhs.subs([
        (uhat(z, mu[j])*vhat(z, mu[j]), solve(h_uvhat_j, uhat(z, mu[j])*vhat(z, mu[j]))[0])
    ]).simplify()
)
duvhat_rho_c = duvhat_rho_b.subs([d_rho_uvhatj.args])
duvhat_rho_c = Eq(duvhat_rho_c.lhs, duvhat_rho_c.rhs.simplify())


duvhat_rho
duvhat_rho_b
duvhat_rho_c

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), Derivative(uhat(z, mu[j])*vhat(z, mu[j])*a[j]/h(z), z))

Eq(Derivative(uhat(z, mu[j])*vhat(z, mu[j])*a[j]/h(z), z), Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z)*a[j]/h(z) - uhat(z, mu[j])*vhat(z, mu[j])*Derivative(h(z), z)*a[j]/h(z)**2)

Eq(Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z), (-uhat(z, mu[j])*vhat(z, mu[j])*Omega[0]*a[j] - a[0])*h(z)*Derivative(rho(z), z)/(a[0]*a[j]))

Eq(Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z), -(Omega[0]*gamma[j] + 1)*h(z)**2*Derivative(rho(z), z)/(a[0]*a[j]))

Eq(Derivative(uhat(z, mu[j])*vhat(z, mu[j]), z), (Omega[0]*gamma[j] + 1)*(Product(uhat(z, mu[j]), (j, 1, N)) - Product(vhat(z, mu[j]), (j, 1, N)))*h(z)**(2 - N/2)*sqrt(Product(a[j], (j, 1, N)))/(a[0]*a[j]))

Suppose the modes satisfy an algebraic identity of the following form in the original coordinates:

In [297]:
H_uv = Eq(
    Sum(b[m]*rho(z)**m,(m,0,M)), 
    Product(u(z, mu[j]),(j,1,N)) + Product(v(z, mu[j]),(j,1,N))
)
H_uv

Eq(Sum(rho(z)**m*b[m], (m, 0, M)), Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))

In [298]:
H_uvhat = Eq(
    Sum(b[m]*rho(z).subs([rho_h_suff.args])**m,(m,0,M)), 
    sqrt(Product(a[j],(j,1,N)))*(
        Product(uhat(z, mu[j]),(j,1,N)) + Product(vhat(z, mu[j]),(j,1,N))
    )/h(z)**(N/2)
)
H_uvhat

Eq(Sum(((-h(z) + a[0])/(h(z)*Omega[0]))**m*b[m], (m, 0, M)), (Product(uhat(z, mu[j]), (j, 1, N)) + Product(vhat(z, mu[j]), (j, 1, N)))*sqrt(Product(a[j], (j, 1, N)))/h(z)**(N/2))

In [323]:
prod_uv_id = Eq(
    (Product(u(z, mu[j]),(j,1,N)) + Product(v(z, mu[j]),(j,1,N)))**2 - 
    (-Product(u(z, mu[j]),(j,1,N)) + Product(v(z, mu[j]),(j,1,N)))**2,
    4*Product(u(z, mu[j])*v(z, mu[j]),(j,1,N))
)
uv_prod_rho_gam = Eq(prod_uv_id.rhs, 4*Product(gamma[j] - rho(z),(j,1,N)))

drho_sqrd = Eq(
    d_rho_uvj.lhs**2,
    d_rho_uvj.rhs**2 - H_uv.rhs**2 + H_uv.lhs**2
)
drho_sqrd_b = drho_sqrd.subs([prod_uv_id.args, uv_prod_rho_gam.args])

drho_sqrd
prod_uv_id
uv_prod_rho_gam
drho_sqrd_b

Eq(Derivative(rho(z), z)**2, (-Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))**2 - (Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))**2 + Sum(rho(z)**m*b[m], (m, 0, M))**2)

Eq(-(-Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))**2 + (Product(u(z, mu[j]), (j, 1, N)) + Product(v(z, mu[j]), (j, 1, N)))**2, 4*Product(u(z, mu[j])*v(z, mu[j]), (j, 1, N)))

Eq(4*Product(u(z, mu[j])*v(z, mu[j]), (j, 1, N)), 4*Product(-rho(z) + gamma[j], (j, 1, N)))

Eq(Derivative(rho(z), z)**2, -4*Product(-rho(z) + gamma[j], (j, 1, N)) + Sum(rho(z)**m*b[m], (m, 0, M))**2)

### Special case N=4, M=2

In [299]:
H_uvhat_N4_M2 = H_uvhat.subs([(N,4),(M,2)])

H_uvhat_N4_M2 = Eq(
    (H_uvhat_N4_M2.lhs*h(z)**2).doit().expand().collect(h(z), factor),
    H_uvhat_N4_M2.rhs*h(z)**2
)

H_uvhat_N4_M2

Eq((Omega[0]*b[1] - 2*b[2])*h(z)*a[0]/Omega[0]**2 + (Omega[0]**2*b[0] - Omega[0]*b[1] + b[2])*h(z)**2/Omega[0]**2 + a[0]**2*b[2]/Omega[0]**2, (Product(uhat(z, mu[j]), (j, 1, 4)) + Product(vhat(z, mu[j]), (j, 1, 4)))*sqrt(Product(a[j], (j, 1, 4))))

In [300]:
h_N4_M2 = duvhat_rho_c.subs([
    uvhat_h_j.args,
    (N,4)
])

h_N4_M2 = Eq(h_N4_M2.lhs.doit().factor(), h_N4_M2.rhs)
_4_2_lhs_coeff = (h_N4_M2.lhs/diff(h(z),z)).simplify()
h_N4_M2 = Eq(
    (h_N4_M2.lhs/_4_2_lhs_coeff).factor(), 
    (h_N4_M2.rhs/_4_2_lhs_coeff).factor()
)

h_N4_M2

Eq(Derivative(h(z), z), (Product(uhat(z, mu[j]), (j, 1, 4)) - Product(vhat(z, mu[j]), (j, 1, 4)))*Omega[0]*sqrt(Product(a[j], (j, 1, 4)))/a[0])

In [302]:
_ham_coeff = Omega[0]/a[0]
h_N4_M2_sqrd = Eq(
    h_N4_M2.lhs**2,
    h_N4_M2.rhs**2 - (H_uvhat_N4_M2.rhs*_ham_coeff)**2 + (H_uvhat_N4_M2.lhs*_ham_coeff)**2
)

h_N4_M2_sqrd_b = Eq(
    h_N4_M2_sqrd.lhs,
    h_N4_M2_sqrd.rhs.doit()
    .expand().collect(h(z), factor)
)

h_N4_M2_sqrd
h_N4_M2_sqrd_b

Eq(Derivative(h(z), z)**2, (Product(uhat(z, mu[j]), (j, 1, 4)) - Product(vhat(z, mu[j]), (j, 1, 4)))**2*Omega[0]**2*Product(a[j], (j, 1, 4))/a[0]**2 - (Product(uhat(z, mu[j]), (j, 1, 4)) + Product(vhat(z, mu[j]), (j, 1, 4)))**2*Omega[0]**2*Product(a[j], (j, 1, 4))/a[0]**2 + ((Omega[0]*b[1] - 2*b[2])*h(z)*a[0]/Omega[0]**2 + (Omega[0]**2*b[0] - Omega[0]*b[1] + b[2])*h(z)**2/Omega[0]**2 + a[0]**2*b[2]/Omega[0]**2)**2*Omega[0]**2/a[0]**2)

Eq(Derivative(h(z), z)**2, 2*(Omega[0]*b[1] - 2*b[2])*(Omega[0]**2*b[0] - Omega[0]*b[1] + b[2])*h(z)**3/(Omega[0]**2*a[0]) + 2*(Omega[0]*b[1] - 2*b[2])*h(z)*a[0]*b[2]/Omega[0]**2 - (4*uhat(z, mu[1])*uhat(z, mu[2])*uhat(z, mu[3])*uhat(z, mu[4])*vhat(z, mu[1])*vhat(z, mu[2])*vhat(z, mu[3])*vhat(z, mu[4])*Omega[0]**4*a[1]*a[2]*a[3]*a[4] - a[0]**4*b[2]**2)/(Omega[0]**2*a[0]**2) + (Omega[0]**2*b[0] - Omega[0]*b[1] + b[2])**2*h(z)**4/(Omega[0]**2*a[0]**2) + (2*Omega[0]**2*b[0]*b[2] + Omega[0]**2*b[1]**2 - 6*Omega[0]*b[1]*b[2] + 6*b[2]**2)*h(z)**2/Omega[0]**2)

In [325]:
uvhat_prod_h_N = Eq(
    Product(uhat(z, mu[j])*vhat(z, mu[j]),(j,1,N)),
    Product((uhat(z, mu[j])*vhat(z, mu[j])).subs([uvhat_h_j.args]),(j,1,N))
)

uvhat_prod_h_N4_M2 = uvhat_prod_h_N.subs(N,4)

uvhat_prod_h_N4_M2 = Eq(
    uvhat_prod_h_N4_M2.lhs.doit(),
    uvhat_prod_h_N4_M2.rhs.doit().expand().collect(h(z), factor)
)

h_N4_M2_sqrd_c = h_N4_M2_sqrd_b.subs([uvhat_prod_h_N4_M2.args])
h_N4_M2_sqrd_c = Eq(h_N4_M2_sqrd_c.lhs, h_N4_M2_sqrd_c.rhs.expand().collect(h(z), factor))


uvhat_prod_h_N
h_N4_M2_sqrd_c

Eq(Product(uhat(z, mu[j])*vhat(z, mu[j]), (j, 1, N)), Product((h(z)*Omega[0]*gamma[j] + h(z) - a[0])/(Omega[0]*a[j]), (j, 1, N)))

Eq(Derivative(h(z), z)**2, (b[2] - 2)*(b[2] + 2)*a[0]**2/Omega[0]**2 + 2*(Omega[0]*b[1]*b[2] + 2*Omega[0]*gamma[1] + 2*Omega[0]*gamma[2] + 2*Omega[0]*gamma[3] + 2*Omega[0]*gamma[4] - 2*b[2]**2 + 8)*h(z)*a[0]/Omega[0]**2 + (2*Omega[0]**2*b[0]*b[2] + Omega[0]**2*b[1]**2 - 4*Omega[0]**2*gamma[1]*gamma[2] - 4*Omega[0]**2*gamma[1]*gamma[3] - 4*Omega[0]**2*gamma[1]*gamma[4] - 4*Omega[0]**2*gamma[2]*gamma[3] - 4*Omega[0]**2*gamma[2]*gamma[4] - 4*Omega[0]**2*gamma[3]*gamma[4] - 6*Omega[0]*b[1]*b[2] - 12*Omega[0]*gamma[1] - 12*Omega[0]*gamma[2] - 12*Omega[0]*gamma[3] - 12*Omega[0]*gamma[4] + 6*b[2]**2 - 24)*h(z)**2/Omega[0]**2 + 2*(Omega[0]**3*b[0]*b[1] + 2*Omega[0]**3*gamma[1]*gamma[2]*gamma[3] + 2*Omega[0]**3*gamma[1]*gamma[2]*gamma[4] + 2*Omega[0]**3*gamma[1]*gamma[3]*gamma[4] + 2*Omega[0]**3*gamma[2]*gamma[3]*gamma[4] - 2*Omega[0]**2*b[0]*b[2] - Omega[0]**2*b[1]**2 + 4*Omega[0]**2*gamma[1]*gamma[2] + 4*Omega[0]**2*gamma[1]*gamma[3] + 4*Omega[0]**2*gamma[1]*gamma[4] + 4*Omega[0]**2*gamma[2]*

In [307]:
h_N4_M2_sqrd_c4_coeff = h_N4_M2_sqrd_c.rhs.expand().coeff(h(z),4).factor()
h_N4_M2_sqrd_c4_coeff_no_b = h_N4_M2_sqrd_c4_coeff.subs([(b[0],0), (b[1],0), (b[2],0)])

h_N4_M2_sqrd_c4_coeff = Eq(
    (h_N4_M2_sqrd_c4_coeff - h_N4_M2_sqrd_c4_coeff_no_b).factor() +
    h_N4_M2_sqrd_c4_coeff_no_b.factor(),
    0
)

h_N4_M2_sqrd_c4_coeff

Eq(-4*(Omega[0]*gamma[1] + 1)*(Omega[0]*gamma[2] + 1)*(Omega[0]*gamma[3] + 1)*(Omega[0]*gamma[4] + 1)/(Omega[0]**2*a[0]**2) + (Omega[0]**2*b[0] - Omega[0]*b[1] + b[2])**2/(Omega[0]**2*a[0]**2), 0)

In [331]:
drho_sqrd_lam_j = Eq(diff(rho(z),z)**2, 4*Product(rho(z) - lam[j],(j,1,4)))

drho_sqrd_b
drho_sqrd_lam_j

Eq(Derivative(rho(z), z)**2, -4*Product(-rho(z) + gamma[j], (j, 1, N)) + Sum(rho(z)**m*b[m], (m, 0, M))**2)

Eq(Derivative(rho(z), z)**2, 4*Product(rho(z) - lambda[j], (j, 1, 4)))

In [332]:
omega_0_no_quartic = Eq(Omega[0], -1/lam[1])
dh_sqrd = drho_sqrd_lam_j.subs([
#     rho_h_suff.args,
    rho_h_suff.subs([omega_0_no_quartic.args]).args
]).doit().simplify()

dh_sqrd = Eq(
    diff(h(z),z)**2,
    solve(dh_sqrd,diff(h(z),z)**2)[0].expand().factor()
)

omega_0_no_quartic
dh_sqrd

Eq(Omega[0], -1/lambda[1])

Eq(Derivative(h(z), z)**2, 4*(-h(z)*lambda[1] + h(z)*lambda[2] + a[0]*lambda[1])*(-h(z)*lambda[1] + h(z)*lambda[3] + a[0]*lambda[1])*(-h(z)*lambda[1] + h(z)*lambda[4] + a[0]*lambda[1])/(a[0]*lambda[1]))

In [329]:
rho_h_suff.subs([omega_0_no_quartic.args])

Eq(rho(z), -(-h(z) + a[0])*lambda[1]/h(z))

### Somethign else

In [336]:
uv_Mob = Eq(u(z)*v(z),(a[0]*U(z)*V(z) + a[1])/(a[2]*U(z)*V(z) + a[3]))
u_Mob = Eq(u(z),(b[0]*U(z) + b[1])/(b[2]*U(z) + b[3]))
v_Mob = Eq(v(z),(c[0]*V(z) + c[1])/(c[2]*V(z) + c[3]))

uv_Mob
u_Mob
v_Mob

Eq(u(z)*v(z), (U(z)*V(z)*a[0] + a[1])/(U(z)*V(z)*a[2] + a[3]))

Eq(u(z), (U(z)*b[0] + b[1])/(U(z)*b[2] + b[3]))

Eq(v(z), (V(z)*c[0] + c[1])/(V(z)*c[2] + c[3]))

In [341]:
uv_Mob_b = Eq((u_Mob.lhs*v_Mob.lhs).subs([uv_Mob.args]), u_Mob.rhs*v_Mob.rhs)
uv_Mob_c = Eq(
    (uv_Mob_eq.lhs - uv_Mob_eq.rhs)*fraction(uv_Mob_eq.lhs)[1]*fraction(uv_Mob_eq.rhs)[1],
    0
)

uv_Mob_b
uv_Mob_c

Eq((U(z)*V(z)*a[0] + a[1])/(U(z)*V(z)*a[2] + a[3]), (U(z)*b[0] + b[1])*(V(z)*c[0] + c[1])/((U(z)*b[2] + b[3])*(V(z)*c[2] + c[3])))

Eq((U(z)*b[2] + b[3])*(V(z)*c[2] + c[3])*(U(z)*V(z)*a[2] + a[3])*(-(U(z)*b[0] + b[1])*(V(z)*c[0] + c[1])/((U(z)*b[2] + b[3])*(V(z)*c[2] + c[3])) + (U(z)*V(z)*a[0] + a[1])/(U(z)*V(z)*a[2] + a[3])), 0)

In [351]:
uv_Mob_d = uv_Mob_c.lhs.simplify().expand().collect([
    (U(z)*V(z))**2, U(z)**2*V(z), U(z)*V(z)**2, U(z)*V(z), U(z), V(z)
], factor)

uv_Mob_d

(a[0]*b[2]*c[2] - a[2]*b[0]*c[0])*U(z)**2*V(z)**2 + (a[0]*b[2]*c[3] - a[2]*b[0]*c[1])*U(z)**2*V(z) + (a[0]*b[3]*c[2] - a[2]*b[1]*c[0])*U(z)*V(z)**2 + (a[1]*b[2]*c[3] - a[3]*b[0]*c[1])*U(z) + (a[1]*b[3]*c[2] - a[3]*b[1]*c[0])*V(z) + (a[0]*b[3]*c[3] + a[1]*b[2]*c[2] - a[2]*b[1]*c[1] - a[3]*b[0]*c[0])*U(z)*V(z) + a[1]*b[3]*c[3] - a[3]*b[1]*c[1]

In [357]:
c_0 = Eq(c[0], solve(uv_Mob_d.coeff((U(z)*V(z))**2),c[0])[0])
c_0

Eq(c[0], a[0]*b[2]*c[2]/(a[2]*b[0]))

In [358]:
uv_Mob_d.subs([c_0.args])

(a[0]*b[2]*c[3] - a[2]*b[0]*c[1])*U(z)**2*V(z) + (a[0]*b[3]*c[2] - a[0]*b[1]*b[2]*c[2]/b[0])*U(z)*V(z)**2 + (a[1]*b[2]*c[3] - a[3]*b[0]*c[1])*U(z) + (-a[0]*a[3]*b[1]*b[2]*c[2]/(a[2]*b[0]) + a[1]*b[3]*c[2])*V(z) + (a[0]*b[3]*c[3] - a[0]*a[3]*b[2]*c[2]/a[2] + a[1]*b[2]*c[2] - a[2]*b[1]*c[1])*U(z)*V(z) + a[1]*b[3]*c[3] - a[3]*b[1]*c[1]